In [ ]:
"""
Double...for when a single column isn't enough

Automated Protein Extraction (APE)

Extract and purify HIS-tagged proteins from E. coli lysates in two columns.

Columns used in the deepwell plate:
    # col 1,7-- Lysate samples in deepwell plate (500 µL supernatant)
    # col 2,8-- 3x binding buffer + mag beads (1000 µL)
    # col 3,9-- Wash buffer 1 (1000 µL)1]
    # col 4,10-- Wash buffer 2 (1000 µL)
    # col 5,11-- Wash buffer 3 (1000 µL)
    # col 6,12-- Empty column to receive eluted proteins (100 µL elution buffer)
r
# Don't forget to set a beginning column for the 1000ul tips below.
e.g. BEGIN_COLUMN = 1; uses 6 columns of tips for 2 columns of samples.
# Don't forget the 96w alpaqua plate [pos0] on rail 7; must be on 3d printed (10mm) supports
# DW plate on [pos1] on rail 19 in plate holder. 

Author : Harley King
Date   : 2025-10-30
Update: 2025-11-14 successfully tested on Hamilton STARlet
Update: 2026-02-22 updated CORE gripper movement
Update: 2026-02-25 doubled my pleasure; works correctly
"""


In [ ]:


# --- Notebook conveniences (safe to ignore when running as a script) ---
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass


import asyncio
from typing import List
import time
import random

In [ ]:



from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import (
    Hamilton_MFX_plateholder_DWP_metal_tapped,
)
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.alpaqua import Alpaqua_96_magnum_flx
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL, # 1000 µL filtered
    hamilton_96_tiprack_10uL_filter, # 10 µL filtered
)
from pylabrobot.liquid_handling.standard import Mix
import time

# --------------------------------------------------------------------------------------
# Deck & labware setup (matches positions/plasticware described in the user snippet)
# --------------------------------------------------------------------------------------
backend = STARBackend()
lh: LiquidHandler = LiquidHandler(backend=backend, deck=STARLetDeck())

deck = STARLetDeck(
  core_grippers="1000uL-at-waste"  # or "1000uL-5mL-on-waste"
) 

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

# No computer or cables are included - unit is as pictured.
tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10


# Magnetic plate on rails=13, module slot 0
# dwp_mod_mag = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_mag")
# car_13 = MFX_CAR_L5_base("car_13", modules={0: dwp_mod_mag})
# lh.deck.assign_child_resource(car_13, rails=13)
# mag_plate = Alpaqua_96_magnum_flx("mag_plate")
# dwp_mod_mag.assign_child_resource(mag_plate)

# Magnetic plate on rails=7, module slot 0
dwp_mod_mag = Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("dwp_mod_mag")
car_7 = MFX_CAR_L5_base("car_7", modules={0: dwp_mod_mag})
lh.deck.assign_child_resource(car_7, rails=7)
mag_plate = Alpaqua_96_magnum_flx("mag_plate")
dwp_mod_mag.assign_child_resource(mag_plate)


# Deep-well plate with samples and reagents on rails=19, module slot 0
dwp_mod_dw = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dw")
car_19 = MFX_CAR_L5_base("car_19", modules={0: dwp_mod_dw})
lh.deck.assign_child_resource(car_19, rails=19)


dw_plate = BioER_96_wellplate_Vb_2200uL("dw_plate")
dwp_mod_dw.assign_child_resource(dw_plate)

await lh.setup(skip_autoload=True)

# STARlet without iSWAP reports 0 arms; but we still want Co-Re gripper moves.
if lh.backend.num_arms == 0:
    lh._resource_pickups = {0: None}  # pragmatic workaround
print(lh.deck.get_resource("core_grippers"))

In [ ]:
# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------
ROWS = ["A","B","C","D","E","F","G","H"]
CHANNELS = list(range(8))  # 8-channel pipetting

async def m_mix(plateCol, mixvol=900): # wells: List, volume: int, repetitions: int = 5):
    well_A1 = dw_plate.get_item("A1")
    lheight = well_A1.compute_height_from_volume(mixvol)
    await lh.aspirate(
            plateCol, 
            vols=[mixvol]*8,
            use_channels=CHANNELS,
            liquid_height = [lheight-1]*8,
            auto_surface_following_distance=True,
            flow_rates=[400]*8,
        )
    await lh.dispense(
            plateCol,
            vols=[mixvol]*8,
            use_channels=CHANNELS,
            liquid_height = [6]*8, # the perfect height for 1000ul. About 1mm when finished. 
            flow_rates=[400]*8,
            mix=[Mix(volume=mixvol, repetitions=2, flow_rate=400)]*8,           
            blow_out=[1]*8, 
            settling_time=[1]*8
        )
async def remove_supernatant(col_from, col_to, vol=1000, final_dispense_height=22, cleanup_vol=200):
    # Decide pass plan (2 passes if > 1000; split evenly)
    if vol > 1000:
        pass_vols = [vol / 2, vol - (vol / 2)]  # e.g., 1500 -> [750, 750]
    else:
        pass_vols = [vol]

    remaining = vol
    for this_pass in pass_vols:
        well_A1 = dw_plate.get_item("A1")
        lheight = well_A1.compute_height_from_volume(remaining)
        await lh.aspirate(
            col_from,
            vols=[this_pass]*8,
            use_channels=CHANNELS,
            liquid_height=[lheight-1]*8,
            auto_surface_following_distance=True,
        )
        remaining = max(0, remaining - this_pass)
        await lh.dispense(
            col_to,
            vols=[this_pass]*8,
            use_channels=CHANNELS,
            liquid_height=[final_dispense_height]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )

    # Optional final sweep for larger volumes.
    if cleanup_vol and vol >= 500:
        await lh.aspirate(
            col_from,
            vols=[cleanup_vol]*8,
            use_channels=CHANNELS,
            liquid_height=[0]*8,
        )
        await lh.dispense(
            col_to,
            vols=[cleanup_vol]*8,
            use_channels=CHANNELS,
            liquid_height=[final_dispense_height]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )
    

async def move_dw_to_mag(gripOne: int, gripTwo: int):
    mag_plate.plate_z_offset = -5.5  # mm

    # STARlet w/out iSWAP: allow PLR resource pickup bookkeeping
    if lh.backend.num_arms == 0:
        lh._resource_pickups = {0: None}

    # core_front_channel is 0-indexed and must be adjacent pair
    if abs(gripOne - gripTwo) != 1:
        raise ValueError("Co-Re gripper channels must be adjacent (e.g., 6 & 7).")
    core_front_channel = max(gripOne, gripTwo)
    if core_front_channel == 0:
        raise ValueError("core_front_channel cannot be 0 (front channel must be >= 1).")

    # ---- MOVE: keep tools mounted so the subsequent push-flush can run ----
    await lh.move_plate(
        plate=dw_plate,
        to=mag_plate,
        use_arm="core",
        core_front_channel=core_front_channel,     # new API :contentReference[oaicite:1]{index=1}
        core_grip_strength=50,
        pickup_distance_from_top=10,
        enable_recovery=True,

        return_core_gripper=False,                 # <-- critical :contentReference[oaicite:2]{index=2}
    )

    # ---- VERIFY + PUSH-FLUSH (requires grippers mounted) ----
    await lh.backend.core_check_resource_exists_at_location_center(
        location=dw_plate.get_absolute_location(),
        resource=dw_plate,
        gripper_y_margin=9,
        enable_recovery=True,
        audio_feedback=False,
    )

    # ---- PARK TOOLS ----
    await lh.backend.return_core_gripper_tools()   # :contentReference[oaicite:3]{index=3}



async def move_dw_back(gripOne: int, gripTwo: int):
    try:
        # ---- STARlet w/out iSWAP: allow PLR resource pickup bookkeeping ----
        if lh.backend.num_arms == 0:
            lh._resource_pickups = {0: None}

        # ---- new API: core_front_channel (0-indexed) ----
        if abs(gripOne - gripTwo) != 1:
            raise ValueError("Co-Re gripper channels must be adjacent (e.g., 6 & 7).")
        core_front_channel = max(gripOne, gripTwo)  # front channel cannot be 0
        if core_front_channel == 0:
            raise ValueError("core_front_channel cannot be 0 (front channel must be >= 1).")

        # ---- MOVE: keep tools mounted if you want push-flush / verification ----
        await lh.move_plate(
            plate=dw_plate,
            to=dwp_mod_dw,
            use_arm="core",
            pickup_distance_from_top=10,
            core_front_channel=core_front_channel,
            core_grip_strength=50,
            enable_recovery=False,

            return_core_gripper=False,  # keep tools mounted so core_check can run
        )
        # recovery not needed for plate as this is hard on grippers
        # # ---- VERIFY + PUSH-FLUSH (optional but recommended) ----
        # await lh.backend.core_check_resource_exists_at_location_center(
        #     location=dw_plate.get_absolute_location(),
        #     resource=dw_plate,
        #     gripper_y_margin=9,
        #     enable_recovery=True,
        #     audio_feedback=False,
        # )

        # ---- PARK TOOLS ----
        await lh.backend.return_core_gripper_tools()

    except Exception as e:
        print("[WARN] Implement gripper move for automated transfer back.")
        print("Exception:", e)




In [ ]:
# --------------------------------------------------------------------------------------
# Main functions
# --------------------------------------------------------------------------------------

def plate_col(col_num: int):
    return dw_plate[f"A{col_num}:H{col_num}"]


def sample_set(base_col: int, tip_col: int, name: str):
    """base_col maps as sample,binding,wash1,wash2,wash3,elution -> base..base+5."""
    return {
        "name": name,
        "tip": tip_col,
        "sample": base_col,
        "binding": base_col + 1,
        "wash1": base_col + 2,
        "wash2": base_col + 3,
        "wash3": base_col + 4,
        "elution": base_col + 5,
    }


async def _pickup_1000(tip_col: int):
    await lh.pick_up_tips(tiprack_1000[f"A{tip_col}:H{tip_col}"], use_channels=CHANNELS)


async def _drop_1000(tip_col: int):
    await lh.drop_tips(tiprack_1000[f"A{tip_col}:H{tip_col}"], use_channels=CHANNELS)


async def _binding_add(set_cfg):
    sample_col = plate_col(set_cfg["sample"])
    binding_col = plate_col(set_cfg["binding"])

    await _pickup_1000(set_cfg["tip"])
    await lh.aspirate(
        binding_col,
        vols=[1000]*8,
        use_channels=CHANNELS,
        liquid_height=[2]*8,
        mix=[Mix(volume=500, repetitions=3, flow_rate=400)]*8,
    )
    await lh.dispense(
        sample_col,
        vols=[1000]*8,
        use_channels=CHANNELS,
        liquid_height=[20]*8,
        blow_out=[1]*8,
        settling_time=[1]*8,
    )
    await lh.aspirate(
        binding_col,
        vols=[100]*8,
        use_channels=CHANNELS,
        liquid_height=[0]*8,
        flow_rates=[100]*8,
    )
    await lh.dispense(
        sample_col,
        vols=[100]*8,
        use_channels=CHANNELS,
        liquid_height=[20]*8,
        flow_rates=[100]*8,
        blow_out=[1]*8,
        settling_time=[1]*8,
    )
    await _drop_1000(set_cfg["tip"])


async def _interleaved_mix(sample_sets, rounds: int, mixvol: int, incubate_seconds: int, label: str):
    for n in range(rounds):
        for set_cfg in sample_sets:
            await _pickup_1000(set_cfg["tip"])
            await m_mix(plate_col(set_cfg["sample"]), mixvol=mixvol)
            await _drop_1000(set_cfg["tip"])
        if n < rounds - 1:
            await asyncio.sleep(incubate_seconds)
        print(f"{label}: round {n+1}/{rounds}")


async def _remove_supernatant_on_magnet(sample_sets, vol: int, final_dispense_height: int = 22, discard=False):
    for set_cfg in sample_sets:
        await _pickup_1000(set_cfg["tip"])
        await remove_supernatant(
            plate_col(set_cfg["sample"]),
            plate_col(set_cfg["target"]),
            vol=vol,
            final_dispense_height=final_dispense_height,
        )
        if discard:
            await lh.discard_tips()
        else:
            await _drop_1000(set_cfg["tip"])


async def add_binding_buffer_and_mix_dual(sample_set_1, sample_set_2, rounds=10, incubate_seconds=90):
    """Binding for both sample sets with interleaved mix/incubation and one magnet move."""
    sample_sets = [sample_set_1, sample_set_2]

    for set_cfg in sample_sets:
        await _binding_add(set_cfg)

    start = time.time()
    await _interleaved_mix(
        sample_sets,
        rounds=rounds,
        mixvol=1000,
        incubate_seconds=incubate_seconds,
        label="Binding mix",
    )
    print(f"Binding mixing and incubation time: {round((time.time() - start)/60, 2)} minutes")

    gripOne = random.randint(1, 6)
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    await asyncio.sleep(120)

    for set_cfg in sample_sets:
        set_cfg["target"] = set_cfg["binding"]
    await _remove_supernatant_on_magnet(sample_sets, vol=1500, discard=True)

    await move_dw_back(gripOne, gripTwo)
    print("Binding step complete for columns 1 and 7.")


async def wash_step_dual(sample_set_1, sample_set_2, wash_key="wash1", rounds=3, incubate_seconds=45):
    """Wash both sample sets using wash1 or wash2 with interleaved mixing."""
    sample_sets = [sample_set_1, sample_set_2]

    for set_cfg in sample_sets:
        sample_col = plate_col(set_cfg["sample"])
        wash_col = plate_col(set_cfg[wash_key])
        await _pickup_1000(set_cfg["tip"])
        await lh.aspirate(
            wash_col,
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height=[20]*8,
            auto_surface_following_distance=True,
        )
        await lh.dispense(
            sample_col,
            vols=[1000]*8,
            use_channels=CHANNELS,
            liquid_height=[10]*8,
            flow_rates=[400]*8,
            mix=[Mix(volume=1000, repetitions=2, flow_rate=400)]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )
        await _drop_1000(set_cfg["tip"])

    await _interleaved_mix(
        sample_sets,
        rounds=rounds,
        mixvol=1000,
        incubate_seconds=incubate_seconds,
        label=f"{wash_key} mix",
    )

    gripOne = random.randint(1, 6)
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    await asyncio.sleep(60)

    for set_cfg in sample_sets:
        set_cfg["target"] = set_cfg[wash_key]
    await _remove_supernatant_on_magnet(sample_sets, vol=1000, discard=False)

    await move_dw_back(gripOne, gripTwo)
    print(f"{wash_key} complete for both sample sets.")


async def final_wash_and_bead_transfer_dual(sample_set_1, sample_set_2, rounds=3, incubate_seconds=45):
    """Final wash and transfer beads to fresh wells for both sample sets."""
    sample_sets = [sample_set_1, sample_set_2]

    for set_cfg in sample_sets:
        sample_col = plate_col(set_cfg["sample"])
        dest_col = plate_col(set_cfg["wash3"])

        await _pickup_1000(set_cfg["tip"])
        await lh.aspirate(
            dest_col,
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height=[1]*8,
        )
        await lh.dispense(
            sample_col,
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height=[2]*8,
            mix=[Mix(volume=500, repetitions=4, flow_rate=300)]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )
        await lh.aspirate(
            sample_col,
            vols=[500]*8,
            use_channels=CHANNELS,
            mix=[Mix(volume=500, repetitions=2, flow_rate=200)]*8,
            liquid_height=[1]*8,
        )
        await lh.dispense(
            dest_col,
            vols=[500]*8,
            use_channels=CHANNELS,
            liquid_height=[10]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )
        await _drop_1000(set_cfg["tip"])

    for n in range(rounds):
        for set_cfg in sample_sets:
            await _pickup_1000(set_cfg["tip"])
            await m_mix(plate_col(set_cfg["wash3"]), mixvol=900)
            await _drop_1000(set_cfg["tip"])
        if n < rounds - 1:
            await asyncio.sleep(incubate_seconds)
        print(f"Final wash mix: round {n+1}/{rounds}")

    gripOne = random.randint(1, 6)
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    await asyncio.sleep(60)

    for set_cfg in sample_sets:
        await _pickup_1000(set_cfg["tip"])
        await remove_supernatant(
            plate_col(set_cfg["wash3"]),
            plate_col(set_cfg["sample"]),
            vol=1000,
        )
        await lh.discard_tips()

    await move_dw_back(gripOne, gripTwo)
    print("Final wash and bead transfer complete for both sample sets.")


async def elution_step_dual(sample_set_1, sample_set_2, incubate_seconds=120):
    """Elute proteins for both sample sets and recover eluate."""
    sample_sets = [sample_set_1, sample_set_2]

    for set_cfg in sample_sets:
        bead_col = plate_col(set_cfg["wash3"])
        elute_col = plate_col(set_cfg["elution"])

        await _pickup_1000(set_cfg["tip"])
        await lh.aspirate(
            elute_col,
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height=[1]*8,
        )
        await lh.dispense(
            bead_col,
            vols=[100]*8,
            use_channels=CHANNELS,
            liquid_height=[1]*8,
            mix=[Mix(volume=100, repetitions=6, flow_rate=50)]*8,
            blow_out=[1]*8,
            settling_time=[1]*8,
        )
        await _drop_1000(set_cfg["tip"])

    await asyncio.sleep(incubate_seconds)

    for set_cfg in sample_sets:
        await _pickup_1000(set_cfg["tip"])
        await m_mix(plate_col(set_cfg["wash3"]), mixvol=100)
        await _drop_1000(set_cfg["tip"])

    gripOne = random.randint(1, 6)
    gripTwo = gripOne + 1
    await move_dw_to_mag(gripOne, gripTwo)
    await asyncio.sleep(60)

    for set_cfg in sample_sets:
        await _pickup_1000(set_cfg["tip"])
        await remove_supernatant(
            plate_col(set_cfg["wash3"]),
            plate_col(set_cfg["elution"]),
            vol=100,
            final_dispense_height=5,
        )
        await lh.discard_tips()

    await move_dw_back(gripOne, gripTwo)
    print("Elution complete for both sample sets.")



In [ ]:
# --------------------------------------------------------------------------------------
# Column mapping
# --------------------------------------------------------------------------------------
# Base mapping for each sample set:
# base+0 -> Lysate samples
# base+1 -> 3x binding buffer + beads
# base+2 -> Wash buffer 1
# base+3 -> Wash buffer 2
# base+4 -> Wash buffer 3
# base+5 -> Elution destination

FIRST_SAMPLESET_TIP_BEGIN_COLUMN = 1 
SECOND_SAMPLESET_TIP_BEGIN_COLUMN = 4 # above int + 3

sample_set_1 = sample_set(base_col=1, tip_col=FIRST_SAMPLESET_TIP_BEGIN_COLUMN, name="set_1_col1")
sample_set_2 = sample_set(base_col=7, tip_col=SECOND_SAMPLESET_TIP_BEGIN_COLUMN, name="set_2_col7")

# Tip usage per function: binding uses tip, washes use tip+1, elution uses tip+2
sample_set_1["tip"] = FIRST_SAMPLESET_TIP_BEGIN_COLUMN
sample_set_2["tip"] = SECOND_SAMPLESET_TIP_BEGIN_COLUMN
await add_binding_buffer_and_mix_dual(sample_set_1, sample_set_2)

sample_set_1["tip"] = FIRST_SAMPLESET_TIP_BEGIN_COLUMN + 1
sample_set_2["tip"] = SECOND_SAMPLESET_TIP_BEGIN_COLUMN + 1
await wash_step_dual(sample_set_1, sample_set_2, wash_key="wash1")
await wash_step_dual(sample_set_1, sample_set_2, wash_key="wash2")
await final_wash_and_bead_transfer_dual(sample_set_1, sample_set_2)

sample_set_1["tip"] = FIRST_SAMPLESET_TIP_BEGIN_COLUMN + 2
sample_set_2["tip"] = SECOND_SAMPLESET_TIP_BEGIN_COLUMN + 2
await elution_step_dual(sample_set_1, sample_set_2)



In [ ]:

# col 1-- Lysate samples in deepwell plate (500ul supernatant)
# col 2-- 3x binding buffer + mag beads (1100ul)
# col 3-- Wash buffer 1 (1500ul)
# col 4-- Wash buffer 2 (1500ul)
# col 5-- Elution buffer (100ul)
# col 6-- Empty column to receive eluted proteins (100ul)
# await lh.backend.return_core_gripper_tools() 

# Function 1: Add 3xBB-b, mix and incubate
#1.1 Mix col 2 containing 1100ul 3x binding buffer with mag beads (3xBB-b) with 8 channels
#1.2 Transfer 1000ul 3xBB-b from col 2 to col 1 containing lysate. 
#1.3 Mix col 1 containing lysate + binding buffer + mag beads. Pause for N minutes.
#1.4 Repeat step 1.3 O times
#1.5 Move dw plate to mag module, pause N minutes, remove supernatant to col 2. 
#1.6 Remove dw plate from mag module and return to original position.

# Function 2: Wash beads 3x
#2.1 Add 1500ul wash buffer from col 3 to col 1. 
#2.2 Mix, pause P minutes; repeat Q times
#2.3 Move dw plate to mag module, pause N minutes, Remove supernatant to col 3
#2.4 Remove dw plate from mag module and return to original position. 
#2.5 Repeat 2.1 to 2.4 for wash buffer in col 4.

# Function 3: Elute proteins
#3.1 Add 100ul elution buffer from col 5 to col 1
#3.2 Mix, pause R minutes; repeat S times
#3.3 Move dw plate to mag module, pause N minutes, transfer eluted proteins from col 1 to col 6.
#3.4 Remove dw plate from mag module and return to original position.
#3.5 End of protocol.

# async def add_binding_buffer():



In [ ]:
# await dispense_mastermix()
# await add_template()

In [ ]:
# await lh.dispense(dw_plate["A1:H1"], vols=[100]*8, liquid_height=[20]*8, use_channels=CHANNELS)
# await lh.drop_tips(tiprack_1000["C7:H7"], use_channels=[2,3,4,5,6,7])
# await lh.drop_tips(tiprack_1000["A6:H6"], use_channels=CHANNELS)
# await lh.drop_tips(tiprack_1000["F6:H6"], use_channels=[5,6,7])
# await lh.discard_tips()
# # await lh.stop()
# await backend.return_core_gripper_tools()